# Reconocimiento de Configuraciones Manuales en la Lengua de Señas Argentina
## Comparación entre Clasificación CNN y Detección con YOLOv8 sobre el Dataset LSA16

---

| Campo | Detalle |
|---|---|
| **Materia** | Redes Neuronales |
| **Integrantes** | Julian Santiago Gonzalez Avendaño, Agustin Garcia Smith |
| **Dataset** | LSA16 — Quiroga et al., UNLP (2016, 2017) |
| **Referencia principal** | Quiroga et al. *"A Study of Convolutional Architectures for Handshape Recognition applied to Sign Language"*, CACIC 2017 |
| **Año** | 2026 |

---

## Resumen

Este trabajo reproduce y extiende los experimentos de Quiroga et al. (2017) sobre reconocimiento de configuraciones manuales (*handshapes*) de la Lengua de Señas Argentina usando el dataset LSA16.

El paper de referencia demostró que la **pre-segmentación de la mano** produce un salto de precisión de +12.64 puntos porcentuales (de 83.54% a 96.18% con LeNet). Sin embargo, ese pipeline requería **condiciones de laboratorio controladas**: guantes fluorescentes, fondo blanco e iluminación uniforme.

Nuestra pregunta de investigación es:

> *¿Es posible reemplazar ese costoso paso de segmentación manual con un modelo de detección de objetos moderno (YOLOv8), habilitando así el reconocimiento en condiciones no controladas?*

Para responderla, diseñamos un experimento en tres etapas:
1. **Replicación del baseline**: Entrenamos LeNetLSA16 sobre las imágenes ya segmentadas del dataset (condición ideal del paper).
2. **Demostración de la degradación**: Entrenamos el mismo modelo sobre las imágenes completas con fondo (condición "raw" del paper).
3. **Propuesta YOLO** *(Fase 3)*: Entrenamos YOLOv8 para detectar y clasificar la mano directamente sobre la imagen completa, sin segmentación previa.


---

## Fase 1: Exploración y Preprocesamiento del Dataset LSA16

### 1.1 Descripción del Dataset

El dataset **LSA16** fue presentado por Ronchetti et al. (2016) y utilizado como benchmark en Quiroga et al. (2017). Contiene las **16 configuraciones manuales (*handshapes*) más frecuentes** del léxico de la LSA.

> **Nota terminológica importante:** Las clases de LSA16 **no son letras del abecedario**. Son *handshapes*, es decir, configuraciones estáticas de la mano que, combinadas con movimiento, forman señas con significado semántico completo.

| Índice | Nombre de la clase | Descripción |
|---|---|---|
| 0 | **Five** | Mano abierta, cinco dedos extendidos |
| 1 | **Four** | Cuatro dedos extendidos |
| 2 | **Horns** | Cuernos / gesto de rock |
| 3 | **Curve** | Mano curva |
| 4 | **Fingers together** | Dedos juntos ("montoncito") |
| 5 | **Double** | Doble |
| 6 | **Hook** | Gancho |
| 7 | **Index** | Solo el dedo índice extendido |
| 8 | **L** | Forma de letra L |
| 9 | **Flat Hand** | Mano plana |
| 10 | **Mitten** | Manopla (dedos juntos, rectos) |
| 11 | **Beak** | Pico |
| 12 | **Thumb** | Pulgar arriba |
| 13 | **Fist** | Puño cerrado |
| 14 | **Telephone** | Gesto de "Shaka" / teléfono |
| 15 | **V** | Forma de V / gesto de paz |

**Composición:** 16 clases × 10 sujetos × 5 repeticiones = **800 imágenes** (Sección 2.1, Quiroga et al. 2017).

**Condiciones de captura:** Los sujetos utilizaron **guantes de colores fluorescentes** sobre fondo blanco con iluminación controlada (Sección 2.1, paper 2016). Las imágenes muestran a la persona desde los hombros hacia arriba, con **ambas manos** realizando la configuración simultáneamente.


### 1.2 Estructura de los archivos del dataset

El dataset contiene dos carpetas:

| Carpeta | Contenido | Tamaño |
|---|---|---|
| `lsa16_raw/` | Fotos originales en color (RGB), 640×480 px | 800 archivos |
| `lsa16_segmented_right_hand/` | Crops de la **mano derecha** ya segmentada, escala de grises | 800 archivos |

**Un detalle clave descubierto en la exploración:** Las imágenes en `lsa16_segmented_right_hand` **no son del mismo tamaño que las raw**. Son *recortes independientes* de la mano derecha, con dimensiones variables (aproximadamente 90–260 px). El nombre del archivo es idéntico al correspondiente en `lsa16_raw`.

Esto tiene dos implicaciones importantes:

1. **Para el experimento "segmented"**: La imagen segmentada de la mano *ya es la imagen recortada*. Solo hay que redimensionarla y usarla directamente como entrada a la CNN.
2. **Para el experimento "raw"**: Usamos la imagen completa 640×480.
3. **Para YOLO**: Necesitamos las coordenadas de la mano **dentro de la imagen raw** (640×480). Las extraemos detectando los guantes fluorescentes por color.

**Decisión de diseño sobre las dos manos:** Dado que el dataset solo provee la segmentación de la mano *derecha* (`lsa16_segmented_right_hand`), utilizamos únicamente esa mano como unidad de clasificación. Esto es consistente con el dataset oficial y con Quiroga et al. (2017).


In [1]:
# ─────────────────────────────────────────────────────────────────────────────
# CELDA 1 — Librerías e importaciones
# ─────────────────────────────────────────────────────────────────────────────
import os
import cv2
import glob
import random
import shutil
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

print("✅ Librerías cargadas.")

# Diccionario de las 16 clases del dataset LSA16 (Quiroga et al., 2017)
LSA16_CLASSES = {
    0:  "Five (mano abierta)",
    1:  "Four (cuatro dedos)",
    2:  "Horns (cuernos)",
    3:  "Curve (mano curva)",
    4:  "Fingers together",
    5:  "Double",
    6:  "Hook (gancho)",
    7:  "Index (índice)",
    8:  "L",
    9:  "Flat Hand (mano plana)",
    10: "Mitten (manopla)",
    11: "Beak (pico)",
    12: "Thumb (pulgar)",
    13: "Fist (puño)",
    14: "Telephone (shaka)",
    15: "V (paz)"
}
LSA16_NAMES = [v.split(' ')[0] for v in LSA16_CLASSES.values()]

RAW_DIR  = 'lsa16_raw'
MASK_DIR = 'lsa16_segmented_right_hand'
OUT_DIR  = 'dataset_yolo'

print(f"   Clases: {LSA16_NAMES}")


✅ Librerías cargadas.
   Clases: ['Five', 'Four', 'Horns', 'Curve', 'Fingers', 'Double', 'Hook', 'Index', 'L', 'Flat', 'Mitten', 'Beak', 'Thumb', 'Fist', 'Telephone', 'V']


### 1.3 Exploración visual del dataset

Antes de cualquier preprocesamiento, visualizamos ejemplos de ambas carpetas para entender la estructura real de los datos. Esto es una práctica de rigor en cualquier proyecto de Machine Learning: nunca asumir el contenido de un dataset sin verificarlo empíricamente.


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELDA 2 — Exploración visual: raw vs. segmentada para 6 clases
# ─────────────────────────────────────────────────────────────────────────────

clases_muestra = [1, 3, 5, 8, 11, 14]
fig, axes = plt.subplots(2, 6, figsize=(18, 6))
fig.suptitle('Fig. 1 — Contenido del dataset LSA16\n'
             'Fila superior: imagen raw (640×480). Fila inferior: mano derecha segmentada (tamaño variable).',
             fontsize=12, fontweight='bold')

axes[0][0].set_ylabel('Raw (lsa16_raw)', fontsize=9, fontweight='bold')
axes[1][0].set_ylabel('Segmentada (lsa16_segmented_right_hand)', fontsize=9, fontweight='bold')

for col, cls in enumerate(clases_muestra):
    fname     = f"{cls}_1_1.png"
    raw_bgr   = cv2.imread(os.path.join(RAW_DIR, fname))
    raw_rgb   = cv2.cvtColor(raw_bgr, cv2.COLOR_BGR2RGB)
    mask      = cv2.imread(os.path.join(MASK_DIR, fname), cv2.IMREAD_GRAYSCALE)
    cls_name  = list(LSA16_CLASSES.values())[cls - 1].split('(')[0].strip()

    axes[0][col].imshow(raw_rgb)
    axes[0][col].set_title(f'Clase {cls}\n"{cls_name}"\n{raw_rgb.shape[1]}×{raw_rgb.shape[0]}px',
                           fontsize=9, fontweight='bold')
    axes[0][col].axis('off')

    axes[1][col].imshow(mask, cmap='gray')
    axes[1][col].set_title(f'{mask.shape[1]}×{mask.shape[0]}px', fontsize=8, color='gray')
    axes[1][col].axis('off')

plt.tight_layout()
plt.savefig('fig1_exploracion_dataset.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Fig. 1 guardada: fig1_exploracion_dataset.png")


### 1.4 Estrategia de preprocesamiento

Como se puede observar en la Fig. 1, las imágenes segmentadas **ya contienen solo la región de la mano derecha**, sin el fondo. Son el resultado del pipeline de normalización descrito en la Sección 2.2 del paper 2016 (rotación canónica, corrección de inversión, recorte). Por lo tanto, **las usamos directamente como entrada a la CNN en el experimento "segmentado"**.

Para el experimento YOLO, necesitamos las coordenadas de la mano dentro de la imagen **raw** (640×480). Las calculamos detectando los guantes fluorescentes por umbralización de color en el espacio HSV: los guantes tienen alta saturación y matiz distintivo, lo que permite aislarlos del fondo blanco con alta fiabilidad.

**Sobre la normalización de coordenadas YOLO:** El formato YOLO espera coordenadas normalizadas por las dimensiones de la **imagen raw** (640×480), no por las de la máscara. Esta distinción es crítica para que YOLO entrene correctamente.


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELDA 3 — Función: detectar bbox del guante en la imagen raw por color HSV
# ─────────────────────────────────────────────────────────────────────────────

def get_bbox_from_raw_hsv(raw_path, saturation_threshold=80):
    """
    Detecta la bounding box del guante fluorescente en la imagen raw (640×480)
    usando umbralización de saturación en el espacio de color HSV.

    El guante fluorescente tiene saturación muy alta (>80 en escala 0-255),
    mientras que el fondo blanco tiene saturación ~0. Esto permite aislar
    la región del guante sin necesidad de la máscara de segmentación.

    Como la persona muestra AMBAS manos, la bbox resultante cubre las dos.
    Para YOLO esto es aceptable: le indicamos dónde está la 'zona de señas'.
    Para la CNN (experimento raw), usamos la imagen completa sin recortar.

    Retorna:
        (xc_norm, yc_norm, w_norm, h_norm) normalizados respecto a 640×480,
        o None si no se detecta guante.
    """
    raw_bgr = cv2.imread(raw_path)
    if raw_bgr is None:
        return None

    hsv = cv2.cvtColor(raw_bgr, cv2.COLOR_BGR2HSV)
    # Máscara de píxeles con saturación alta (guante fluorescente)
    sat_mask = (hsv[:, :, 1] > saturation_threshold).astype(np.uint8) * 255

    # Operaciones morfológicas para rellenar huecos y eliminar ruido pequeño
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (5, 5))
    sat_mask = cv2.morphologyEx(sat_mask, cv2.MORPH_CLOSE, kernel)
    sat_mask = cv2.morphologyEx(sat_mask, cv2.MORPH_OPEN, kernel)

    y_coords, x_coords = np.where(sat_mask > 0)
    if len(y_coords) == 0:
        return None

    x_min, x_max = x_coords.min(), x_coords.max()
    y_min, y_max = y_coords.min(), y_coords.max()

    h_img, w_img = raw_bgr.shape[:2]
    xc = (x_min + x_max) / 2 / w_img
    yc = (y_min + y_max) / 2 / h_img
    w  = (x_max - x_min) / w_img
    h  = (y_max - y_min) / h_img

    return xc, yc, w, h


print("✅ Función de detección por color HSV definida.")


### 1.5 Verificación visual del método de detección

Verificamos que el método HSV detecta correctamente la región de los guantes en varias clases antes de aplicarlo masivamente.


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELDA 4 — Verificación visual de la detección HSV en 6 ejemplos
# ─────────────────────────────────────────────────────────────────────────────

fig, axes = plt.subplots(2, 6, figsize=(18, 6))
fig.suptitle('Fig. 2 — Verificación del método HSV para detección de guantes\n'
             'Fila superior: imagen raw con bbox detectada. Fila inferior: máscara de saturación.',
             fontsize=12, fontweight='bold')

axes[0][0].set_ylabel('Raw + BBox detectada', fontsize=9, fontweight='bold')
axes[1][0].set_ylabel('Máscara de saturación HSV', fontsize=9, fontweight='bold')

for col, cls in enumerate(clases_muestra):
    fname    = f"{cls}_1_1.png"
    raw_path = os.path.join(RAW_DIR, fname)
    raw_bgr  = cv2.imread(raw_path)
    raw_rgb  = cv2.cvtColor(raw_bgr, cv2.COLOR_BGR2RGB)
    cls_name = list(LSA16_CLASSES.values())[cls - 1].split('(')[0].strip()

    hsv      = cv2.cvtColor(raw_bgr, cv2.COLOR_BGR2HSV)
    sat_mask = (hsv[:, :, 1] > 80).astype(np.uint8) * 255
    bbox     = get_bbox_from_raw_hsv(raw_path)

    axes[0][col].imshow(raw_rgb)
    axes[0][col].set_title(f'Clase {cls} "{cls_name}"', fontsize=9, fontweight='bold')
    axes[0][col].axis('off')
    if bbox:
        xc, yc, w, h = bbox
        h_img, w_img = raw_bgr.shape[:2]
        bx = (xc - w/2) * w_img
        by = (yc - h/2) * h_img
        bw = w * w_img
        bh = h * h_img
        rect = patches.Rectangle((bx, by), bw, bh,
                                  linewidth=2.5, edgecolor='#00FF41', facecolor='none')
        axes[0][col].add_patch(rect)
        axes[0][col].set_xlabel(f'xc={xc:.2f} yc={yc:.2f} w={w:.2f} h={h:.2f}',
                                fontsize=7, color='#005500', fontfamily='monospace')

    axes[1][col].imshow(sat_mask, cmap='gray')
    axes[1][col].axis('off')

plt.tight_layout()
plt.savefig('fig2_verificacion_hsv.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Fig. 2 guardada: fig2_verificacion_hsv.png")


Si la figura muestra los bounding boxes (verde) envolviendo correctamente ambas manos en todos los ejemplos, el método HSV es válido para generar las anotaciones de YOLO.

### 1.6 Pipeline de preprocesamiento completo

Ejecutamos el pipeline sobre las 800 imágenes para generar el dataset en formato YOLO.

**División del dataset:** Usamos una partición fija 70% / 15% / 15% con semilla `42` para reproducibilidad. A diferencia del esquema de validación cruzada de Quiroga et al. (2017), elegimos una partición fija porque:
1. Permite comparar CNN y YOLOv8 sobre **exactamente los mismos datos**.
2. Provee un conjunto de validación independiente del test para monitorear el entrenamiento.
3. Es el estándar en la comunidad de deep learning moderno (cf. LeCun et al., 2015).


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELDA 5 — Pipeline completo: detección HSV + división del dataset
# ─────────────────────────────────────────────────────────────────────────────

if os.path.exists(OUT_DIR):
    shutil.rmtree(OUT_DIR)
for split in ['train', 'val', 'test']:
    os.makedirs(os.path.join(OUT_DIR, 'images', split), exist_ok=True)
    os.makedirs(os.path.join(OUT_DIR, 'labels', split), exist_ok=True)
    os.makedirs(os.path.join(OUT_DIR, 'segmented', split), exist_ok=True)

raw_files = sorted(glob.glob(os.path.join(RAW_DIR, '*.png')))
random.seed(42)
random.shuffle(raw_files)

total     = len(raw_files)
train_end = int(total * 0.70)
val_end   = int(total * 0.85)

split_map = {}
for i, fp in enumerate(raw_files):
    if i < train_end:   split_map[fp] = 'train'
    elif i < val_end:   split_map[fp] = 'val'
    else:               split_map[fp] = 'test'

counts  = {'train': 0, 'val': 0, 'test': 0}
skipped = 0

for raw_path, split in split_map.items():
    filename  = os.path.basename(raw_path)
    mask_path = os.path.join(MASK_DIR, filename)

    bbox = get_bbox_from_raw_hsv(raw_path)
    if bbox is None:
        skipped += 1
        continue

    xc, yc, w, h = bbox
    class_idx = int(filename.split('_')[0]) - 1

    # Etiqueta YOLO (coordenadas normalizadas respecto a la imagen raw 640×480)
    label_path = os.path.join(OUT_DIR, 'labels', split, filename.replace('.png', '.txt'))
    with open(label_path, 'w') as f:
        f.write(f"{class_idx} {xc:.6f} {yc:.6f} {w:.6f} {h:.6f}\n")

    # Imagen raw (para experimento B y para YOLO)
    shutil.copy(raw_path, os.path.join(OUT_DIR, 'images', split, filename))

    # Imagen segmentada (para experimento A — CNN sobre mano recortada)
    if os.path.exists(mask_path):
        shutil.copy(mask_path, os.path.join(OUT_DIR, 'segmented', split, filename))

    counts[split] += 1

total_ok = sum(counts.values())
print("=" * 54)
print("  PREPROCESAMIENTO COMPLETADO")
print("=" * 54)
print(f"  Imágenes procesadas  : {total_ok}")
print(f"  Descartadas (sin HSV): {skipped}")
print(f"  {'─'*40}")
print(f"  Train : {counts['train']:>4}  ({counts['train']/total_ok*100:.1f}%)")
print(f"  Val   : {counts['val']:>4}  ({counts['val']/total_ok*100:.1f}%)")
print(f"  Test  : {counts['test']:>4}  ({counts['test']/total_ok*100:.1f}%)")
print("=" * 54)


### 1.7 Análisis de la distribución de clases por split

Verificamos que la distribución de clases es uniforme en los tres subconjuntos. Una distribución desequilibrada introduciría sesgo en los resultados.


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELDA 6 — Distribución de clases por split
# ─────────────────────────────────────────────────────────────────────────────

split_counts = {s: [0]*16 for s in ['train', 'val', 'test']}
for split in ['train', 'val', 'test']:
    for fname in os.listdir(os.path.join(OUT_DIR, 'labels', split)):
        if fname.endswith('.txt'):
            cls = int(fname.split('_')[0]) - 1
            split_counts[split][cls] += 1

x      = np.arange(16)
w_bar  = 0.25
colors = ['#2196F3', '#FF9800', '#4CAF50']
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

ax = axes[0]
for i, (split, color) in enumerate(zip(['train', 'val', 'test'], colors)):
    ax.bar(x + i*w_bar, split_counts[split], w_bar, label=split.capitalize(),
           color=color, alpha=0.85)
ax.set_xticks(x + w_bar)
ax.set_xticklabels(LSA16_NAMES, rotation=40, ha='right', fontsize=8)
ax.set_xlabel('Configuración manual')
ax.set_ylabel('Cantidad de imágenes')
ax.set_title('Distribución por clase y split')
ax.legend()
ax.grid(axis='y', alpha=0.3)

totales = [sum(split_counts[s]) for s in ['train', 'val', 'test']]
axes[1].pie(totales,
    labels=[f'Train\n({totales[0]})', f'Val\n({totales[1]})', f'Test\n({totales[2]})'],
    colors=colors, autopct='%1.1f%%', startangle=90, textprops={'fontsize': 11})
axes[1].set_title('Proporción global Train / Val / Test')

plt.suptitle('Fig. 3 — Distribución del dataset LSA16 (partición 70/15/15)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig3_distribucion_dataset.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"✅ Fig. 3 guardada. Total: {sum(totales)} | Train: {totales[0]} | Val: {totales[1]} | Test: {totales[2]}")


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELDA 7 — Generación de dataset.yaml para YOLOv8
# ─────────────────────────────────────────────────────────────────────────────

class_names_yaml = [v.split('(')[0].strip() for v in LSA16_CLASSES.values()]

yaml_content = f"""path: ../{OUT_DIR}
train: images/train
val:   images/val
test:  images/test

nc: 16
names: {class_names_yaml}
"""

yaml_path = os.path.join(OUT_DIR, 'dataset.yaml')
with open(yaml_path, 'w') as f:
    f.write(yaml_content)

print("✅ dataset.yaml generado:")
for line in yaml_content.strip().split('\n'):
    print(f"   {line}")


---

## Fase 2: Experimento Comparativo con Redes Neuronales Convolucionales (CNN)

### 2.1 Contexto académico y estado del arte

Quiroga et al. (2017) evaluaron múltiples arquitecturas CNN sobre LSA16 usando imágenes pre-segmentadas. Sus resultados son la **línea de base** contra la cual comparamos:

| Arquitectura | Accuracy en LSA16 (%) |
|---|---|
| ProbSom — método clásico (Ronchetti et al., 2016) | 92.30 |
| **LeNet** (LeCun et al., 1998) | **95.78** |
| AllConvolutional (Springenberg et al., 2014) | 94.56 |
| **VGG16** (Simonyan & Zisserman, 2014) | **95.92** |
| ResNet-34 (He et al., 2016) | 93.49 |

*(Tabla 1, Quiroga et al., 2017 — todos sobre imágenes segmentadas)*

El resultado más relevante para nuestro trabajo es la **Tabla 2** del mismo paper, que compara distintos esquemas de preprocesamiento usando LeNet:

| Esquema de preprocesamiento | Accuracy LeNet (%) |
|---|---|
| **Raw — imagen completa con fondo** | **83.54** |
| **Segmented hand — mano recortada, RGB** | **96.18** |
| Grayscale | 87.08 |
| Black & White | 91.38 |

*La diferencia entre raw y segmented es de **+12.64 puntos porcentuales**, incluso en condiciones controladas. Esto motiva la búsqueda de un segmentador automático.*

### 2.2 Hipótesis experimentales

**H1 (Baseline segmentado):** Una CNN entrenada sobre las imágenes ya segmentadas del dataset (mano derecha recortada, escala de grises) alcanzará un accuracy similar al 95–96% reportado por Quiroga et al. (2017), validando que nuestra implementación es correcta.

**H2 (Degradación por fondo):** La misma CNN entrenada sobre las imágenes completas (640×480, con fondo) presentará una caída de accuracy consistente con los 83.54% del paper.

> **Nota metodológica:** El experimento "segmentado" de Quiroga et al. usaba imágenes RGB a color. Las imágenes segmentadas de LSA16 que tenemos son en escala de grises (la máscara provee la forma, no el color). Esto puede producir una ligera diferencia en los resultados absolutos respecto al paper, pero no afecta la validez del experimento comparativo entre las dos condiciones.


### 2.3 Librerías de Deep Learning (PyTorch)

Utilizamos el framework **PyTorch** (Paszke et al., 2019), el estándar de facto en investigación académica de deep learning por su flexibilidad, transparencia y amplia documentación.


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELDA 8 — Librerías de Deep Learning (PyTorch)
# ─────────────────────────────────────────────────────────────────────────────
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
from PIL import Image
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
import seaborn as sns

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ PyTorch {torch.__version__} cargado.")
print(f"   Dispositivo: {device}")
if device.type == 'cuda':
    print(f"   GPU: {torch.cuda.get_device_name(0)}")


### 2.4 Dataset con dos modos de entrada: `segmented` y `raw`

Creamos una clase `Dataset` de PyTorch que soporta los dos modos del experimento:

- **`mode='segmented'`**: carga directamente la imagen de `lsa16_segmented_right_hand/` (mano ya recortada, escala de grises). Replica la condición "Segmented Hand" del paper.
- **`mode='raw'`**: carga la imagen completa de `dataset_yolo/images/`. Replica la condición "Raw" del paper.

Usar la misma clase con distinto `mode` garantiza que **ninguna otra variable cambia** entre los dos experimentos, lo que hace la comparación científicamente válida.


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELDA 9 — Clase Dataset personalizada para LSA16
# ─────────────────────────────────────────────────────────────────────────────

class LSA16Dataset(Dataset):
    """
    Dataset PyTorch para los experimentos comparativos sobre LSA16.

    Modos:
        'segmented': carga la mano ya segmentada (dataset_yolo/segmented/{split}/).
                     La clase se extrae del nombre del archivo (1-indexed → 0-indexed).
        'raw'      : carga la imagen completa (dataset_yolo/images/{split}/).
    """

    def __init__(self, split, mode='segmented', transform=None):
        self.mode      = mode
        self.transform = transform

        if mode == 'segmented':
            self.img_dir = os.path.join(OUT_DIR, 'segmented', split)
        else:
            self.img_dir = os.path.join(OUT_DIR, 'images', split)

        self.files = sorted([f for f in os.listdir(self.img_dir) if f.endswith('.png')])

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        fname    = self.files[idx]
        img_path = os.path.join(self.img_dir, fname)

        # La clase está codificada en el nombre: "ClaseID_SujetoID_RepID.png"
        # Ejemplo: "3_7_2.png" → clase 3 (1-indexed) → 2 (0-indexed)
        class_id = int(fname.split('_')[0]) - 1

        # .convert('RGB') convierte grayscale a 3 canales idénticos,
        # necesario para la arquitectura CNN que espera 3 canales de entrada.
        image = Image.open(img_path).convert('RGB')

        if self.transform:
            image = self.transform(image)

        return image, class_id


print("✅ LSA16Dataset definida.")
print("   Modo 'segmented' → imagen de mano recortada (grayscale → RGB)")
print("   Modo 'raw'       → imagen completa 640×480 con fondo")


### 2.5 Transformaciones

**Tamaño de entrada:** 128×128 px, estándar del paper (Quiroga et al., 2017).

**Por qué NO usamos RandomHorizontalFlip:**
El dataset LSA16 tiene una convención fija de color: la mano **derecha** siempre lleva el guante **rojo**. Si volteamos horizontalmente la imagen de la mano derecha recortada, parece una mano izquierda (que no está en las imágenes segmentadas). Para las imágenes raw, voltear intercambia la posición del guante rojo y magenta, generando señales contradictorias. Por ello, sólo aplicamos `RandomRotation(10°)` como augmentation mínima.

Nota: En la sección de replicación exacta del paper (2.8), usamos **cero augmentation** para máxima fidelidad al protocolo original.


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELDA 10 — Pipelines de transformación
# ─────────────────────────────────────────────────────────────────────────────

IMG_SIZE = 128  # Quiroga et al. (2017), Sección 2.2

# Para LOSO (replicación exacta del paper): sin augmentation
loso_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

# Para nuestro approach (partición fija): solo rotación suave, SIN flip horizontal
# (el flip cambia la quiralidad de la mano y la posición de los guantes)
train_transform_seg = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

train_transform_raw = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

print(f"✅ Transformaciones definidas. Entrada: {IMG_SIZE}×{IMG_SIZE} px")
print(f"   loso_transform:      resize + normalize (sin augmentation)")
print(f"   train_transform_*:   resize + rotation(10°) + normalize (sin flip)")


### 2.6 Arquitectura: LeNetLSA16

Implementamos la variante de **LeNet** utilizada por Quiroga et al. (2017) para el dataset LSA16. LeNet fue originalmente propuesta por LeCun et al. (1998) para reconocimiento de caracteres escritos a mano, y es una de las primeras redes convolucionales de la historia.

**¿Por qué LeNet?** Aunque existen arquitecturas más modernas (VGG16, ResNet), LeNet ofrece un balance ideal para datasets pequeños: suficiente capacidad de representación sin el riesgo de sobreajuste que tendría una red más profunda con solo 560 imágenes de entrenamiento. Esto está empíricamente respaldado por Quiroga et al. (2017), donde LeNet logró 95.78% frente al 95.92% de VGG16, siendo VGG16 órdenes de magnitud más costosa computacionalmente.

**Diferencias con la LeNet original (LeCun et al., 1998):**

| Componente | LeNet original | Nuestra implementación | Justificación / Fuente |
|---|---|---|---|
| Activación | Tanh / Sigmoid | **ELU** | Evita neuronas muertas, gradientes más suaves (Clevert et al., 2015) |
| Normalización | Ninguna | **BatchNorm2d** | Estabiliza y acelera el entrenamiento (Ioffe & Szegedy, 2015) |
| Profundidad | 2 bloques conv | **4 bloques conv** | Mayor capacidad para 16 clases con imágenes 128×128 |
| Regularización | Ninguna | **Dropout(0.5)** | Previene el sobreajuste desactivando neuronas al azar (Srivastava et al., 2014) |

**Flujo de datos a través de la red:**

Cada **bloque convolucional** aplica tres operaciones en secuencia:
- **`Conv2d`**: aplica un conjunto de filtros aprendibles (por ejemplo, 32 filtros de 3×3) para detectar patrones locales en la imagen. Cada filtro genera un "mapa de características" que responde a un patrón específico (bordes, texturas, etc.).
- **`ELU`**: función de activación no lineal. Sin ella, toda la red sería equivalente a una transformación lineal sin importar cuántas capas tenga. ELU (Exponential Linear Unit) es preferible a ReLU porque produce valores negativos en lugar de ceros para entradas negativas, lo que mejora el flujo del gradiente.
- **`BatchNorm2d`**: normaliza las activaciones de cada batch para que tengan media ≈ 0 y varianza ≈ 1. Esto evita que los valores crezcan o decrezcan descontroladamente a medida que pasan por capas sucesivas.
- **`MaxPool2d(2,2)`**: reduce la resolución espacial a la mitad tomando el valor máximo de cada ventana 2×2. Esto reduce el número de parámetros y hace al modelo más robusto a pequeñas traslaciones.

```
Entrada:   [3 × 128 × 128]   ← imagen RGB redimensionada
Bloque 1:  [32 × 64 × 64]   ← detecta bordes y gradientes de color
Bloque 2:  [64 × 32 × 32]   ← detecta curvas y formas simples
Bloque 3:  [128 × 16 × 16]  ← detecta partes de la mano (dedos, nudillos)
Bloque 4:  [256 × 8 × 8]    ← detecta configuraciones completas de la mano
Flatten:   16384 valores
Dense 1:   512 neuronas + Dropout(0.5)
Salida:    16 neuronas  ← una puntuación (logit) por clase
```


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELDA 11 — Arquitectura LeNetLSA16
# Replica el baseline de Quiroga et al. (2017)
# Componentes referenciados en: Clevert et al. (2015), Ioffe & Szegedy (2015),
#                               Srivastava et al. (2014), LeCun et al. (1998)
# ─────────────────────────────────────────────────────────────────────────────

class LeNetLSA16(nn.Module):

    def __init__(self, num_classes=16):
        super(LeNetLSA16, self).__init__()

        self.features = nn.Sequential(
            # ── Bloque 1: detecta bordes y gradientes de color ────────────────
            # padding=1 mantiene el tamaño espacial antes del MaxPool
            nn.Conv2d(3, 32, kernel_size=3, padding=1),   # [3,128,128] → [32,128,128]
            nn.ELU(),
            nn.BatchNorm2d(32),
            nn.MaxPool2d(2, 2),                           # → [32, 64, 64]

            # ── Bloque 2: detecta curvas, ángulos y formas simples ────────────
            nn.Conv2d(32, 64, kernel_size=3, padding=1),  # → [64, 64, 64]
            nn.ELU(),
            nn.BatchNorm2d(64),
            nn.MaxPool2d(2, 2),                           # → [64, 32, 32]

            # ── Bloque 3: detecta partes de la mano (dedos, nudillos) ─────────
            nn.Conv2d(64, 128, kernel_size=3, padding=1), # → [128, 32, 32]
            nn.ELU(),
            nn.BatchNorm2d(128),
            nn.MaxPool2d(2, 2),                           # → [128, 16, 16]

            # ── Bloque 4: detecta configuraciones completas de la mano ────────
            nn.Conv2d(128, 256, kernel_size=3, padding=1),# → [256, 16, 16]
            nn.ELU(),
            nn.BatchNorm2d(256),
            nn.MaxPool2d(2, 2),                           # → [256,  8,  8]
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),                        # [256, 8, 8] → vector de 16384 valores
            nn.Linear(256 * 8 * 8, 512),         # combinación lineal aprendida
            nn.ELU(),
            nn.Dropout(0.5),                     # regularización: desactiva 50% de neuronas
            nn.Linear(512, num_classes)          # 16 puntuaciones finales (logits)
        )

    def forward(self, x):
        return self.classifier(self.features(x))


# Verificación con tensor sintético
_m   = LeNetLSA16()
_x   = torch.zeros(1, 3, IMG_SIZE, IMG_SIZE)
_o   = _m(_x)
n_p  = sum(p.numel() for p in _m.parameters())
print("✅ LeNetLSA16 verificada.")
print(f"   Entrada  : {list(_x.shape)}")
print(f"   Salida   : {list(_o.shape)}  (16 logits)")
print(f"   Parámetros: {n_p:,}")
del _m, _x, _o


### 2.7 Función de entrenamiento

**Decisiones de hiperparámetros y su justificación:**

| Hiperparámetro | Valor | Justificación |
|---|---|---|
| Optimizador | **Adam** | Adapta el LR por parámetro, más robusto que SGD (Kingma & Ba, 2015). Mismo que Quiroga et al. (2017) |
| Learning Rate | **0.0007** | Valor de referencia del paper |
| Weight decay | **1e-4** | Regularización L2 adicional para reducir sobreajuste en datasets pequeños |
| Batch size | **32** | Balance entre estabilidad del gradiente y velocidad |
| Épocas | **50** | Suficiente para convergencia. El modo raw requiere más épocas al ser una tarea más difícil |
| Loss function | **CrossEntropyLoss** | Estándar para clasificación multiclase |

La función imprime métricas **en cada época** para facilitar el diagnóstico del entrenamiento.


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELDA 12 — Función de entrenamiento con registro completo por época
# ─────────────────────────────────────────────────────────────────────────────

def train_model(model, train_loader, val_loader, epochs=50, lr=0.0007,
                weight_decay=1e-4, label="Modelo"):
    # weight_decay (L2): penaliza pesos grandes, reduciendo el sobreajuste.
    # Especialmente importante para el modo raw donde hay pocos ejemplos.
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    criterion = nn.CrossEntropyLoss()
    history   = {k: [] for k in ['train_loss','val_loss','train_acc','val_acc']}

    print(f"\n{'='*64}")
    print(f"  {label}")
    print(f"  Épocas: {epochs} | LR: {lr} | WD: {weight_decay} | Device: {device}")
    print(f"{'='*64}")
    print(f"  {'Época':>5}  {'Train Loss':>10}  {'Train Acc':>9}  {'Val Loss':>9}  {'Val Acc':>8}")
    print(f"  {'─'*50}")

    for epoch in range(epochs):
        # ── Entrenamiento ─────────────────────────────────────────────────────
        model.train()
        t_loss, t_ok, t_n = 0.0, 0, 0
        for X, y in train_loader:
            X, y = X.to(device), y.to(device)
            optimizer.zero_grad()
            out  = model(X)
            loss = criterion(out, y)
            loss.backward()
            optimizer.step()
            t_loss += loss.item() * X.size(0)
            t_ok   += out.argmax(1).eq(y).sum().item()
            t_n    += y.size(0)
        tl, ta = t_loss / t_n, t_ok / t_n

        # ── Validación ────────────────────────────────────────────────────────
        model.eval()
        v_loss, v_ok, v_n = 0.0, 0, 0
        with torch.no_grad():
            for X, y in val_loader:
                X, y = X.to(device), y.to(device)
                out  = model(X)
                v_loss += criterion(out, y).item() * X.size(0)
                v_ok   += out.argmax(1).eq(y).sum().item()
                v_n    += y.size(0)
        vl, va = v_loss / v_n, v_ok / v_n

        for k, v in zip(['train_loss','val_loss','train_acc','val_acc'], [tl, vl, ta, va]):
            history[k].append(v)

        print(f"  {epoch+1:>5}  {tl:>10.4f}  {ta:>9.4f}  {vl:>9.4f}  {va:>8.4f}")

    print(f"{'='*64}")
    print(f"  Accuracy final en validación: {va:.4f}  ({va*100:.2f}%)")
    return history

print("✅ Función de entrenamiento definida.")


### 2.8 Replicación Exacta del Paper: Cross-Validación Leave-One-Subject-Out (LOSO)

Antes de cualquier extensión metodológica, verificamos que nuestra implementación de LeNetLSA16 reproduce los resultados del paper de referencia. Esto es un principio fundamental del método científico: **no se puede extender lo que no se puede replicar**.

**Protocolo LOSO (Leave-One-Subject-Out):**

El dataset LSA16 tiene **10 sujetos** numerados del 1 al 10. En cada fold de LOSO:
- **Entrenamiento**: imágenes de los 9 sujetos restantes (720 imágenes)
- **Test**: imágenes del sujeto excluido (80 imágenes: 16 clases × 5 repeticiones)

Este proceso se repite 10 veces, una por sujeto. Se reporta el accuracy promedio ± desviación estándar sobre los 10 folds. Esta estrategia garantiza que el modelo nunca ve imágenes del mismo sujeto en train y test, evaluando la **generalización a nuevos sujetos**.

**Por qué LOSO y no nuestra partición fija:** Quiroga et al. (2017) casi certamente usaron LOSO dado que es el protocolo estándar en reconocimiento de lengua de señas con datasets de pocos sujetos. Con LOSO, cada fold entrena con 720 imágenes (vs. 560 en nuestra partición fija), lo que reduce el sobreajuste y mejora la generalización.

**Preprocesamiento:** Sólo resize a 128×128 + normalización. Sin augmentation, para máxima fidelidad al protocolo del paper.

> ⏱️ *Tiempo estimado: 40–80 min (CPU) / 5–10 min (GPU). Cada uno de los 10 folds entrena un modelo completo desde cero.*


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELDA LOSO-1 — Dataset auxiliar para LOSO (carga desde lista de archivos)
# ─────────────────────────────────────────────────────────────────────────────

class FileListDataset(Dataset):
    # Dataset que carga imágenes desde una lista explícita de rutas.
    # Extrae la clase del nombre del archivo: {clase}_{sujeto}_{rep}.png

    def __init__(self, files, transform=None):
        self.files     = files
        self.transform = transform

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        path     = self.files[idx]
        fname    = os.path.basename(path)
        class_id = int(fname.split('_')[0]) - 1  # 1-indexed → 0-indexed
        image    = Image.open(path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, class_id


def run_loso_cv(img_dir, experiment_name, ref_acc, epochs=40, lr=0.0007):
    # Ejecuta LOSO cross-validation.
    # Para cada sujeto de test (1-10): entrena con los 9 restantes, evalúa en el excluido.
    all_files = sorted(glob.glob(os.path.join(img_dir, '*.png')))
    if not all_files:
        print(f"⚠️  No se encontraron archivos en {img_dir}")
        return [], [], []

    print(f"\n{'='*64}")
    print(f"  LOSO — {experiment_name}")
    print(f"  Archivos totales: {len(all_files)} | Folds: 10 | Épocas/fold: {epochs}")
    print(f"  Referencia paper: {ref_acc}%")
    print(f"{'='*64}")
    print(f"  {'Fold':>4}  {'Sujeto':>7}  {'Train':>6}  {'Test':>5}  {'Acc':>8}")
    print(f"  {'-'*40}")

    fold_accs           = []
    all_preds_global    = []
    all_labels_global   = []

    for test_subj in range(1, 11):
        train_files = [f for f in all_files
                       if int(os.path.basename(f).split('_')[1]) != test_subj]
        test_files  = [f for f in all_files
                       if int(os.path.basename(f).split('_')[1]) == test_subj]

        ds_train = FileListDataset(train_files, transform=loso_transform)
        ds_test  = FileListDataset(test_files,  transform=loso_transform)
        dl_train = DataLoader(ds_train, batch_size=32, shuffle=True,  num_workers=0)
        dl_test  = DataLoader(ds_test,  batch_size=32, shuffle=False, num_workers=0)

        # Modelo fresco por fold (importante: no compartir pesos entre folds)
        model_fold = LeNetLSA16(num_classes=16).to(device)
        opt        = optim.Adam(model_fold.parameters(), lr=lr)
        crit       = nn.CrossEntropyLoss()

        model_fold.train()
        for epoch in range(epochs):
            for X, y in dl_train:
                X, y = X.to(device), y.to(device)
                opt.zero_grad()
                loss = crit(model_fold(X), y)
                loss.backward()
                opt.step()

        # Evaluación
        model_fold.eval()
        preds, labels = [], []
        with torch.no_grad():
            for X, y in dl_test:
                X, y = X.to(device), y.to(device)
                preds.extend(model_fold(X).argmax(1).cpu().numpy())
                labels.extend(y.cpu().numpy())

        fold_acc = accuracy_score(labels, preds)
        fold_accs.append(fold_acc)
        all_preds_global.extend(preds)
        all_labels_global.extend(labels)

        print(f"  {test_subj:>4}  {test_subj:>7}  {len(train_files):>6}  "
              f"{len(test_files):>5}  {fold_acc*100:>7.2f}%")

    mean_acc   = np.mean(fold_accs)
    std_acc    = np.std(fold_accs)
    global_acc = accuracy_score(all_labels_global, all_preds_global)

    print(f"  {'-'*40}")
    print(f"  Accuracy media (por fold): {mean_acc*100:.2f}% ± {std_acc*100:.2f}%")
    print(f"  Accuracy global (800 imgs): {global_acc*100:.2f}%")
    print(f"  Referencia paper:           {ref_acc}%")
    print(f"{'='*64}")

    return fold_accs, all_labels_global, all_preds_global


print("✅ FileListDataset y run_loso_cv definidos.")


#### Experimento LOSO-A: Imágenes segmentadas

Executamos LOSO sobre la carpeta `lsa16_segmented_right_hand/` (mano derecha con guante rojo, sobre fondo negro, imágenes RGB).

Cada uno de los 10 folds entrena un modelo LeNetLSA16 desde cero con los 9 sujetos y lo evalua en el sujeto excluido. No hay data augmentation.


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELDA LOSO-2 — LOSO para imágenes segmentadas
# Ref: "Segmented Hand RGB" → 96.18% (Quiroga et al., 2017, Tabla 2)
# ─────────────────────────────────────────────────────────────────────────────

loso_accs_seg, loso_labels_seg, loso_preds_seg = run_loso_cv(
    MASK_DIR,
    experiment_name='Segmented RGB',
    ref_acc=96.18,
    epochs=40,
    lr=0.0007
)


#### Experimento LOSO-B: Imágenes raw (imagen completa)


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELDA LOSO-3 — LOSO para imágenes raw
# Ref: "Raw" → 83.54% (Quiroga et al., 2017, Tabla 2)
# ─────────────────────────────────────────────────────────────────────────────

loso_accs_raw, loso_labels_raw, loso_preds_raw = run_loso_cv(
    RAW_DIR,
    experiment_name='Raw (imagen completa)',
    ref_acc=83.54,
    epochs=40,
    lr=0.0007
)


#### Resultados LOSO: comparación con el paper


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELDA LOSO-4 — Visualización y tabla de resultados LOSO
# ─────────────────────────────────────────────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
folds = list(range(1, 11))

for ax, accs, label, ref, color in [
    (axes[0], loso_accs_seg, 'Segmented RGB', 96.18, '#2196F3'),
    (axes[1], loso_accs_raw, 'Raw',           83.54, '#F44336'),
]:
    mean_a = np.mean(accs) * 100
    ax.bar(folds, [a*100 for a in accs], color=color, alpha=0.7, edgecolor='white')
    ax.axhline(y=mean_a, color=color, ls='-',  lw=2.5, label=f'Nuestra media: {mean_a:.1f}%')
    ax.axhline(y=ref,    color='black', ls='--', lw=1.5, label=f'Paper: {ref}%')
    ax.set_title(f'LOSO — {label}', fontsize=12, fontweight='bold')
    ax.set_xlabel('Sujeto de Test'); ax.set_ylabel('Accuracy (%)')
    ax.set_ylim(0, 105); ax.set_xticks(folds)
    ax.legend(fontsize=9); ax.grid(axis='y', alpha=0.3)

plt.suptitle('Fig. 5 — Resultados LOSO (replicación del protocolo de Quiroga et al., 2017)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('fig5_loso_results.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n" + "="*68)
print("  TABLA LOSO — Comparación con Quiroga et al. (2017), Tabla 2")
print("="*68)
if loso_accs_seg:
    ms, ss = np.mean(loso_accs_seg)*100, np.std(loso_accs_seg)*100
    gs     = accuracy_score(loso_labels_seg, loso_preds_seg)*100
    print(f"  Segmented: {ms:.2f}% ± {ss:.2f}%  (global: {gs:.2f}%)  | Paper: 96.18%")
if loso_accs_raw:
    mr, sr = np.mean(loso_accs_raw)*100, np.std(loso_accs_raw)*100
    gr     = accuracy_score(loso_labels_raw, loso_preds_raw)*100
    print(f"  Raw:       {mr:.2f}% ± {sr:.2f}%  (global: {gr:.2f}%)  | Paper: 83.54%")
print("="*68)
print()
print("ℹ️  Si la replicación es exitosa (~96% / ~83%), la arquitectura e hiperparámetros")
print("   son correctos. La diferencia con nuestra partición fija (Sec. 2.9–10) se debe")
print("   al protocolo de evaluación (LOSO vs. split fijo) y al mayor tamaño de entrenamiento.")


---

### 2.9 Experimento A (Partición Fija) — CNN sobre imágenes segmentadas

Una vez validada la implementación con LOSO, evaluamos nuestro protocolo de **partición fija 70/15/15** (sección 1.6) sobre el mismo modelo. Esta condición es la que usaremos para la comparación posterior con YOLOv8, ya que ambos modelos se evalúan sobre el mismo conjunto de test.

Las imágenes de `lsa16_segmented_right_hand/` son **RGB a color**: guante rojo sobre fondo negro. El modo `segmented` del `LSA16Dataset` las carga desde `dataset_yolo/segmented/{split}/`, que son copias idénticas.

> **Referencia:** *Segmented Hand RGB* → **96.18%** (Quiroga et al., 2017, Tabla 2).

> **Expectativa:** Con solo 560 imágenes de entrenamiento vs. 720 en LOSO, el resultado será inferior, pero coherente con la tendencia.

> ⏱️ *Tiempo: 10–20 min (CPU) / <3 min (GPU).*


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELDA 13 — Exp. 2.9: CNN (partición fija) sobre imágenes segmentadas
# Ref: "Segmented Hand RGB" → 96.18% (Quiroga et al., 2017, Tabla 2)
# ─────────────────────────────────────────────────────────────────────────────

BATCH_SIZE = 32
EPOCHS     = 50

ds_seg_train = LSA16Dataset('train', mode='segmented', transform=train_transform_seg)
ds_seg_val   = LSA16Dataset('val',   mode='segmented', transform=eval_transform)
dl_seg_train = DataLoader(ds_seg_train, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
dl_seg_val   = DataLoader(ds_seg_val,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f"Train: {len(ds_seg_train)} | Val: {len(ds_seg_val)}")
print(f"Referencia: 96.18% (Segmented Hand RGB, Quiroga et al. 2017, Tabla 2)")

model_seg   = LeNetLSA16(num_classes=16).to(device)
history_seg = train_model(model_seg, dl_seg_train, dl_seg_val,
                          epochs=EPOCHS, lr=0.0007, weight_decay=1e-4,
                          label="Exp. 2.9 — Segmentada (partición fija)")


### 2.10 Experimento B (Partición Fija) — CNN sobre imágenes completas con fondo

Misma arquitectura e hiperparámetros que el Experimento A. La única variable es la fuente de imágenes: imagen completa 640×480, comprimida a 128×128.

**Dificultad intrínseca:** En 128×128, la mano ocupa ∼25–40 px. La CNN debe aprender a ignorar el cuerpo y el fondo, y enfocarse en la configuración del guante (rojo a la derecha, magenta a la izquierda). Esta es la razón fundamental por la que el paper reportó una caída de ~12 puntos porcentuales respecto a la condición segmentada.

> **Referencia:** *Raw* → **83.54%** (Quiroga et al., 2017, Tabla 2).

> ⏱️ *Tiempo estimado similar al experimento anterior.*


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELDA 14 — Exp. 2.10: CNN (partición fija) sobre imagen raw
# Ref: "Raw" → 83.54% (Quiroga et al., 2017, Tabla 2)
# ─────────────────────────────────────────────────────────────────────────────

ds_raw_train = LSA16Dataset('train', mode='raw', transform=train_transform_raw)
ds_raw_val   = LSA16Dataset('val',   mode='raw', transform=eval_transform)
dl_raw_train = DataLoader(ds_raw_train, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
dl_raw_val   = DataLoader(ds_raw_val,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f"Train: {len(ds_raw_train)} | Val: {len(ds_raw_val)}")
print(f"Referencia: 83.54% (Raw, Quiroga et al. 2017, Tabla 2)")

model_raw   = LeNetLSA16(num_classes=16).to(device)
history_raw = train_model(model_raw, dl_raw_train, dl_raw_val,
                          epochs=EPOCHS, lr=0.0007, weight_decay=1e-4,
                          label="Exp. 2.10 — Raw imagen completa (partición fija)")


### 2.10 Análisis de curvas de entrenamiento

Las **líneas punteadas** representan los valores de referencia del paper (Quiroga et al., 2017, Tabla 2).
- Si nuestras curvas convergen hacia esas referencias: nuestra implementación es correcta.
- Si la brecha Train–Val es grande: hay sobreajuste → el modelo memoriza en lugar de generalizar.


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELDA 15 — Curvas de entrenamiento comparativas con referencias del paper
# ─────────────────────────────────────────────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
epocas = range(1, EPOCHS + 1)

ax = axes[0]
ax.plot(epocas, [v*100 for v in history_seg['val_acc']], 'b-', lw=2, label='CNN Segmentada (Val)')
ax.plot(epocas, [v*100 for v in history_raw['val_acc']], 'r-', lw=2, label='CNN Raw/Fondo (Val)')
ax.axhline(y=96.18, color='blue', ls='--', alpha=0.5, lw=1.5, label='Paper: Segmented 96.18%')
ax.axhline(y=83.54, color='red',  ls='--', alpha=0.5, lw=1.5, label='Paper: Raw 83.54%')
ax.set_title('Accuracy en Validación', fontsize=12, fontweight='bold')
ax.set_xlabel('Épocas');  ax.set_ylabel('Accuracy (%)')
ax.legend(fontsize=9);  ax.grid(True, alpha=0.3);  ax.set_ylim(0, 105)

ax2 = axes[1]
ax2.plot(epocas, [v*100 for v in history_seg['train_acc']], 'b--', alpha=0.6, lw=1.5, label='Segmentada (Train)')
ax2.plot(epocas, [v*100 for v in history_seg['val_acc']],   'b-',  lw=2,     label='Segmentada (Val)')
ax2.plot(epocas, [v*100 for v in history_raw['train_acc']], 'r--', alpha=0.6, lw=1.5, label='Raw (Train)')
ax2.plot(epocas, [v*100 for v in history_raw['val_acc']],   'r-',  lw=2,     label='Raw (Val)')
ax2.set_title('Train vs Val — diagnóstico de sobreajuste', fontsize=12, fontweight='bold')
ax2.set_xlabel('Épocas');  ax2.set_ylabel('Accuracy (%)')
ax2.legend(fontsize=9);  ax2.grid(True, alpha=0.3);  ax2.set_ylim(0, 105)

plt.suptitle('Fig. 4 — Curvas de entrenamiento: Exp. A (segmentada) vs Exp. B (raw)\n'
             'Líneas punteadas: valores de Quiroga et al. (2017), Tabla 2',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('fig4_curvas_entrenamiento.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Fig. 4 guardada: fig4_curvas_entrenamiento.png")


### 2.11 Evaluación final sobre el conjunto de Test

El conjunto de test fue **reservado desde el inicio** y no se utilizó en ninguna decisión de diseño ni ajuste de hiperparámetros.


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELDA 16 — Evaluación final en Test + Matrices de Confusión
# ─────────────────────────────────────────────────────────────────────────────

def evaluate_on_test(model, mode, label):
    ds = LSA16Dataset('test', mode=mode, transform=eval_transform)
    dl = DataLoader(ds, batch_size=32, shuffle=False)

    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for X, y in dl:
            X, y = X.to(device), y.to(device)
            all_preds.extend(model(X).argmax(1).cpu().numpy())
            all_labels.extend(y.cpu().numpy())

    acc = accuracy_score(all_labels, all_preds)

    print(f"\n{'='*64}")
    print(f"  RESULTADOS EN TEST — {label}")
    print(f"{'='*64}")
    print(f"  Accuracy global: {acc:.4f}  ({acc*100:.2f}%)")
    print()
    print(classification_report(all_labels, all_preds, target_names=LSA16_NAMES, digits=3))

    cm = confusion_matrix(all_labels, all_preds)
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=LSA16_NAMES, yticklabels=LSA16_NAMES)
    plt.title(f'Matriz de Confusión — {label}', fontsize=12, fontweight='bold')
    plt.ylabel('Clase Real');  plt.xlabel('Clase Predicha')
    plt.tight_layout()
    fname = f"confusion_{mode}.png"
    plt.savefig(fname, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"✅ Guardada: {fname}")
    return acc

acc_seg = evaluate_on_test(model_seg, 'segmented', 'CNN — Imágenes Segmentadas (Exp. A)')
acc_raw = evaluate_on_test(model_raw, 'raw',       'CNN — Imagen Completa Raw (Exp. B)')


### 2.12 Tabla comparativa de resultados vs. estado del arte

Comparamos directamente con los resultados de Quiroga et al. (2017), Tabla 2, usando la misma arquitectura (LeNet) y las mismas condiciones de preprocesamiento.


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELDA 17 — Tabla resumen comparativa con Quiroga et al. (2017)
# ─────────────────────────────────────────────────────────────────────────────

print("=" * 72)
print("  TABLA COMPARATIVA — LeNetLSA16 sobre LSA16")
print("  Referencia: Quiroga et al. (2017), Tabla 2")
print("=" * 72)
print(f"  {'Condición':<40} {'Este trabajo':>12} {'Quiroga 2017':>12}")
print(f"  {'─'*66}")
print(f"  {'Exp. A — CNN Segmentada (RGB color)':<40} {acc_seg*100:>11.2f}%  {'96.18%':>11}")
print(f"  {'Exp. B — CNN Raw / imagen completa':<40} {acc_raw*100:>11.2f}%  {'83.54%':>11}")
print(f"  {'─'*66}")
deg_nuestra = acc_seg*100 - acc_raw*100
print(f"  {'Degradación Seg→Raw (Δ)':<40} {deg_nuestra:>11.2f}%  {'12.64%':>11}")
print("=" * 72)
print()
if acc_seg >= 0.88:
    print(f"  ✅ Exp. A: {acc_seg*100:.1f}% (dentro del rango esperado ~96%)")
elif acc_seg >= 0.75:
    print(f"  ⚠️  Exp. A: {acc_seg*100:.1f}% (por debajo del esperado; diferencia metodológica: split único vs CV)")
else:
    print(f"  ❌ Exp. A: {acc_seg*100:.1f}% (muy por debajo — revisar pipeline)")
if acc_raw >= 0.60:
    print(f"  ✅ Exp. B: {acc_raw*100:.1f}% (dentro del rango esperado ~83.54%)")
elif acc_raw >= 0.40:
    print(f"  ⚠️  Exp. B: {acc_raw*100:.1f}% (por debajo; probablemente requiere más épocas o mayor regularización)")
else:
    print(f"  ❌ Exp. B: {acc_raw*100:.1f}% (muy por debajo — revisar pipeline)")
if deg_nuestra > 5:
    print(f"  ✅ Degradación Seg→Raw: {deg_nuestra:.1f}pp (consistente con {12.64:.2f}pp del paper)")


---

## Próximo paso: Fase 3 — Detección y clasificación con YOLOv8

En la **Fase 3** entrenaremos **YOLOv8** (Jocher et al., 2023), que aborda el problema de raíz: en lugar de recibir una imagen con la mano pre-segmentada, el modelo localiza la mano directamente en la imagen completa y la clasifica en un único *forward pass*, sin necesidad de guantes fluorescentes ni fondos controlados.
